# LifeMate: dự đoán và đo thời gian từng mẫu
Các bước theo phụ lục của hướng dẫn Docker: (1) seed, (2) nạp mô hình và tài nguyên, (3) đọc test case, (4) dự đoán từng mẫu.
Notebook dùng đúng code và cấu hình của `predict.sh` (`/code/flags.sh`, đọc `/code/configs/release.sh`), nên kết quả giống `/result/submission.jsonl`. Kết quả ghi vào `/result/jupyter_submission.jsonl`, mỗi dòng có trường `time` (ms).

In [ ]:
# Bước 1: seed cố định
import os, random, numpy as np, torch
def seed_everything(seed=42):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
seed_everything(42)

In [ ]:
# Bước 2: nạp mô hình (vLLM server cục bộ với base + LoRA; Gipformer ASR trên CPU). Không dùng API ngoài.
import subprocess, time, urllib.request, sys
sys.path.insert(0, '/code/src')
flags = dict(l.split('=', 1) for l in subprocess.run(['bash', '-c', 'source /code/flags.sh; env'], capture_output=True, text=True).stdout.splitlines() if '=' in l)
os.environ.update({k: v for k, v in flags.items() if k.startswith('LIFEMATE_')})   # mô hình, ASR, deadline, prompt, retry cap, vote
t0 = time.perf_counter()
server = subprocess.Popen(['bash', '-c', 'source /code/flags.sh; vllm_server'],
                          stdout=open('/tmp/vllm.log', 'w'), stderr=subprocess.STDOUT)
while True:
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2); break
    except Exception:
        assert server.poll() is None, open('/tmp/vllm.log').read()[-3000:]
        time.sleep(5)
from pathlib import Path
from lifemate.docker_predict import Predictor
pred = Predictor(Path('/data'), 'http://127.0.0.1:8000/v1/chat/completions', os.environ['LIFEMATE_SERVED_MODEL'],
                 Path(os.environ['LIFEMATE_ASR_DIR']), float(os.environ['LIFEMATE_DEADLINE']))
print(f'load model: {time.perf_counter() - t0:.1f} s, prompt {pred.orchestration}, retry cap {pred.retry_cap}')

In [ ]:
# Bước 3: đọc test case (zaic_tools của split được import từ /data/<split> bên trong Predictor)
import json
samples = pred.samples()
print(pred.split, len(samples), 'mẫu')

In [ ]:
# Bước 4: dự đoán từng mẫu, time (ms) đo từ trước khi đọc audio
Path('/result').mkdir(parents=True, exist_ok=True)
with open('/result/jupyter_submission.jsonl', 'w', encoding='utf-8') as fout:
    for item in samples:
        row = pred.predict(item)
        fout.write(json.dumps(row, ensure_ascii=False) + '\n'); fout.flush()
times = [json.loads(l)['time'] for l in open('/result/jupyter_submission.jsonl')]
print(f'predict: {len(times)} mẫu, trung bình {sum(times) / len(times):.0f} ms/mẫu, lâu nhất {max(times)} ms')
server.terminate()